# Optimizador AdaGrad

(Ejercicio, 2 puntos posibles)

AdaGrad adapta la tasa de aprendizaje **por coordenada**. Acumula el cuadrado de los gradientes en un vector $G$ y divide el paso de cada parámetro por $\sqrt{G}$. Las direcciones con gradientes grandes (el eje estrecho del valle) se frenan; las direcciones suaves conservan un paso útil.

En este ejercicio hay dos tareas:

1. **(1 punto)** Implementar el descenso por gradiente con AdaGrad.
2. **(1 punto)** Configurar `learning_rate` y `max_iterations` para acercarte al óptimo $(0, 0)$ por debajo del umbral. Puedes usar hasta 2000 iteraciones.

El valle elíptico, la visualización y el ciclo `optimize` ya están dados.

INSTRUCCIONES: Completa el código faltante.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irvingvasquez/cv2course_intro_nn/blob/master/23_opt_adagrad.ipynb)

@juan1rving


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.animation as animation
import nni


In [ ]:
# --- Criterio de evaluación (fijo: no lo modifiques) ---
# Punto inicial
initial_params = [-4.0, 0.4]
# Distancia euclidiana máxima al óptimo (0, 0)
umbral = 0.1
# Tope de iteraciones para la evaluación
max_iteraciones_permitidas = 2000


In [ ]:
# TODO (1 punto): configura learning_rate y max_iterations
# para acercarte al óptimo. Se considera que llegaste si, en no más
# de 2000 pasos, la distancia a (0, 0) es menor que umbral.
# GD y AdaGrad usarán estos mismos valores.
learning_rate = 0.01
max_iterations = 100


## Problema

Usaremos un **valle elíptico**: una cuadrática con curvaturas muy distintas. Un eje es casi plano y el otro es muy empinado. Ahí se nota AdaGrad, porque cada coordenada necesita un paso de tamaño distinto.

$$
f(x, y) = \tfrac{1}{2}\bigl(a x^{2} + b y^{2}\bigr)
$$

$$
\frac{\partial f}{\partial x} = a x, \qquad \frac{\partial f}{\partial y} = b y
$$

con $a = 1$ y $b = 40$. El mínimo global está en $(0, 0)$ y $f(0,0)=0$.

El descenso por gradiente clásico zigzaguea en $y$ (gradiente grande) y avanza despacio en $x$. AdaGrad reduce el paso en $y$ y deja que $x$ siga avanzando.


In [ ]:
problem = nni.EllipticValleyProblem()
punto_inicial = list(initial_params)

def square_view(xs, ys, margin=0.20):
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    dx = x_max - x_min
    dy = y_max - y_min
    span = max(dx, dy, 1e-6)
    pad_x = margin * (dx if dx > 0 else span)
    pad_y = margin * (dy if dy > 0 else span)
    x_min, x_max = x_min - pad_x, x_max + pad_x
    y_min, y_max = y_min - pad_y, y_max + pad_y
    side = max(x_max - x_min, y_max - y_min)
    cx = 0.5 * (x_min + x_max)
    cy = 0.5 * (y_min + y_max)
    half = side / 2.0
    return cx - half, cx + half, cy - half, cy + half

min_x, max_x, min_y, max_y = square_view(
    [punto_inicial[0], problem.optimal_params[0]],
    [punto_inicial[1], problem.optimal_params[1]],
)

x = np.linspace(min_x, max_x, 200)
y = np.linspace(min_y, max_y, 200)
X, Y = np.meshgrid(x, y)
Z = problem.function(X, Y)

max_iterations_run = min(int(max_iterations), max_iteraciones_permitidas)
gd = nni.GradientDescentOptimizer(
    learning_rate=learning_rate,
    max_iterations=max_iterations_run,
    tolerance=1e-6,
)
gd.optimize(
    problem.function,
    lambda x, y: [problem.gradient_x(x, y), problem.gradient_y(x, y)],
    initial_params,
)
gd_x = [log['params'][0] for log in gd.get_history()]
gd_y = [log['params'][1] for log in gd.get_history()]
gd_z = [log['value'] for log in gd.get_history()]


In [ ]:
# Create a figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# Plot the contour plot in the first subplot
ax1.contourf(X, Y, Z, levels=50, cmap='viridis')
ax1.plot(punto_inicial[0], punto_inicial[1], 'wo', markeredgecolor='k')
ax1.plot(problem.optimal_params[0], problem.optimal_params[1], 'ro')
ax1.plot(gd_x, gd_y, 'C1.-', markersize=5, label=f'GD, lr={gd.learning_rate}, iter={len(gd_x)}')
ax1.set_xlim(min_x, max_x)
ax1.set_ylim(min_y, max_y)
ax1.set_aspect('equal', adjustable='box')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('Contour Plot')
ax1.legend(loc='upper right')

# Plot the 3D projection in the second subplot
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(X, Y, Z, cmap='viridis', alpha=0.7)
ax2.plot(gd_x, gd_y, gd_z, 'C1.-', markersize=5, label='GD')
ax2.scatter(punto_inicial[0], punto_inicial[1], problem.function(*punto_inicial),
            c='w', s=40, edgecolors='k')
ax2.scatter(problem.optimal_params[0], problem.optimal_params[1], 0, c='r', s=40)
ax2.set_xlim(min_x, max_x)
ax2.set_ylim(min_y, max_y)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('z')
ax2.set_title('3D Projection')

plt.tight_layout()
plt.show()


## Actualización AdaGrad

El descenso por gradiente clásico usa el mismo $\eta$ en todas las coordenadas:

$$
\theta_{t} = \theta_{t-1} - \eta \nabla f(\theta_{t-1})
$$

AdaGrad guarda un acumulador $G$ (un valor por parámetro) y adapta el paso:

$$
G_{t} = G_{t-1} + g_{t} \odot g_{t}
$$

$$
\theta_{t} = \theta_{t-1} - \frac{\eta}{\sqrt{G_{t}} + \varepsilon} \odot g_{t}
$$

$\varepsilon$ evita dividir entre cero; usa $10^{-8}$. La clase `nni.GradientDescentOptimizer` ya implementa el ciclo `optimize`. Tú debes completar `AdaGradOptimizer`: inicializar $G$ y sobrescribir `next_params`.


In [ ]:
class AdaGradOptimizer(nni.GradientDescentOptimizer):
    def __init__(self, learning_rate=0.01, max_iterations=1000, tolerance=1e-6, epsilon=1e-8):
        super().__init__(learning_rate=learning_rate, max_iterations=max_iterations, tolerance=tolerance)
        self.epsilon = epsilon
        self.name = 'AdaGrad'
        # TODO (1 punto): implementa AdaGrad.
        # Inicializa el acumulador G (misma dimensión que los parámetros: x, y).
        self.G = None

    def next_params(self, gradient, params):
        # Completa la actualización:
        # 1) Suma el cuadrado del gradiente a G (elemento a elemento).
        # 2) Da el paso restando (learning_rate / (sqrt(G) + epsilon)) * gradiente.
        return None


In [ ]:
max_iterations_run = min(int(max_iterations), max_iteraciones_permitidas)
if max_iterations > max_iteraciones_permitidas:
    print(f'Aviso: max_iterations se recortó a {max_iteraciones_permitidas}.')

optimizer = AdaGradOptimizer(
    learning_rate=learning_rate,
    max_iterations=max_iterations_run,
    tolerance=1e-6,
)

optimized_params = optimizer.optimize(
    problem.function,
    lambda x, y: [problem.gradient_x(x, y), problem.gradient_y(x, y)],
    initial_params,
)

history = optimizer.get_history()
k = min(len(history), max_iteraciones_permitidas)
params_eval = np.array(history[k - 1]['params'], dtype=float)
optimo = np.array(problem.optimal_params, dtype=float)
distancia = np.linalg.norm(params_eval - optimo)
llego = np.isfinite(distancia) and distancia < umbral

print(f'Parámetros evaluados: {params_eval}')
print(f'Óptimo conocido: {optimo}')
print(f'Mejor valor encontrado: {problem.function(*params_eval)}')
print(f'Distancia al óptimo: {distancia:.4f}  (umbral = {umbral})')
print(f'Iteraciones: {k}  (máximo permitido = {max_iteraciones_permitidas})')
if llego:
    print('Evaluación: SÍ se acerca al óptimo.')
else:
    print('Evaluación: NO se acerca al óptimo. Ajusta learning_rate y max_iterations.')


In [ ]:
# Comparación visual: mismo learning_rate e iteraciones
gd = nni.GradientDescentOptimizer(
    learning_rate=learning_rate,
    max_iterations=max_iterations_run,
    tolerance=1e-6,
)
gd.optimize(
    problem.function,
    lambda x, y: [problem.gradient_x(x, y), problem.gradient_y(x, y)],
    initial_params,
)

def path_xy(opt):
    h = opt.get_history()
    return [log['params'][0] for log in h], [log['params'][1] for log in h]

gd_x, gd_y = path_xy(gd)
ada_x, ada_y = path_xy(optimizer)

fig, ax = plt.subplots(figsize=(6, 6))
ax.contourf(X, Y, Z, levels=50, cmap='viridis')
ax.plot(punto_inicial[0], punto_inicial[1], 'wo', markeredgecolor='k')
ax.plot(problem.optimal_params[0], problem.optimal_params[1], 'ro')
ax.plot(gd_x, gd_y, 'C1.-', markersize=5, label=f'GD, lr={gd.learning_rate}, iter={len(gd_x)}')
ax.plot(ada_x, ada_y, 'w.-', markersize=5, label=f'{optimizer.name}, lr={optimizer.learning_rate}, iter={len(ada_x)}')
ax.set_xlim(min_x, max_x)
ax.set_ylim(min_y, max_y)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('GD vs AdaGrad (mismo lr)')
ax.legend(loc='upper right')
plt.show()


In [ ]:
# Animate and save the AdaGrad path
fig, ax = plt.subplots(figsize=(6, 6))
ax.contourf(X, Y, Z, levels=50, cmap='viridis')
ax.plot(ada_x[0], ada_y[0], 'wo', markeredgecolor='k')
ax.plot(problem.optimal_params[0], problem.optimal_params[1], 'ro')
(line,) = ax.plot([], [], 'w.-', markersize=8)
ax.set_xlim(min_x, max_x)
ax.set_ylim(min_y, max_y)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('x')
ax.set_ylabel('y')
title = ax.set_title('')

def update(frame):
    line.set_data(ada_x[:frame + 1], ada_y[:frame + 1])
    title.set_text(f'{optimizer.name} lr = {optimizer.learning_rate}, iter {frame}/{len(ada_x)}')
    return (line, title)

animation.FuncAnimation(fig, update, frames=len(ada_x), interval=30, blit=True, repeat=False).save('adagrad.mp4')
